In [ ]:
#| default_exp augmentations

# Augmentations

> Because this will help

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F
from torch.distributions.beta import Beta


## Patching

In [ ]:
#| export 
def create_patch(xb, patch_len, stride, constant_pad=False, constant_pad_value=0):
    """
    xb: [bs x n_vars x seq_len]
    out: [bs x num_patch x n_vars x patch_len]
    """
    if xb.dim() == 2:
        xb = xb.unsqueeze(0)
    if constant_pad:
        seq_len = xb.shape[-1]
        # seq_len > patch_len and stride <= seq_len - patch_len
        if ((seq_len-patch_len) % stride != 0):
            # only pad if remainder
            xb = F.pad(xb, (0, stride), 'constant', value=constant_pad_value) # pad at the end with value
    xb = xb.permute(0,2,1) # xb: [bs x tgt_len x nvars]
    xb = xb.unfold(dimension=1, size=patch_len, step=stride) # xb: [bs x num_patch x n_vars x patch_len]
    return xb

In [ ]:
#| export
def unpatch(x, seq_len, remove_padding=True):
    """
    x: [bs/None x patch_num x n_vars x patch_len]
    returns x: [bs x n_vars x seq_len]
    """
    if x.dim() == 3:
        x = x.transpose(0,1)
    else:
        x = x.transpose(1,2)
    
    x = x.flatten(start_dim=-2, end_dim=-1)
    if remove_padding:
        x = x[...,:seq_len]
    return x

In [ ]:
#| export
def mask_patches_simple(xb, mask_ratio):
    """
    Function that masks patches using fixed ratio approach similar to random_masking
    
    xb: [bs x patch_num x n_vars x patch_len] or nested tensor
    Returns:
        x_masked: masked tensor with same shape as input
        mask: binary mask where 1 indicates masked positions (bs x patch_num x n_vars)
    """
    if xb.dim() == 3:
        xb = xb.unsqueeze(0)
    x = xb.clone()
    
    bs, L, nvars, D = x.shape
    len_keep = int(L * (1 - mask_ratio))
    
    # Generate noise and sort for masking
    noise = torch.rand(bs, L, nvars, device=xb.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)
    
    # Keep the first subset
    ids_keep = ids_shuffle[:, :len_keep]
    x_kept = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, 1, D))
    
    # Create and combine with zeros for masked positions
    x_removed = torch.zeros(bs, L - len_keep, nvars, D, device=xb.device).detach()
    x_ = torch.cat([x_kept, x_removed], dim=1)
    
    # Restore original order
    x_masked = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, 1, D))
    
    return x_masked

## Value Augmentations

In [ ]:
#| export
def remove_values(x, mask_ratio):
    mask = torch.rand(x.shape, device=x.device) > mask_ratio # maskout_ratio are False, note that this is per channel masking
    x = x*mask
    return x

def jitter_augmentation(x, mask_ratio=0.05, jitter_ratio=0.05, p=1):
    prob = torch.rand(1) < p
    if not prob:
        return x
    x = x.clone()  
    max_amplitude = x.abs().max()  # Use absolute max for amplitude scale

    zero_mask = torch.rand(x.shape, device=x.device) > mask_ratio
    jitter_mask = torch.rand(x.shape, device=x.device) > (1-jitter_ratio)
    jitter_values = torch.randn(x.shape, device=x.device) * max_amplitude * 0.1  # Normal distribution centered at 0 times 10% of max amplitude
    
    x = x*zero_mask + jitter_mask*jitter_values
    return x

## Shuffle Augmentations

In [ ]:
#| export
def shuffle_dim(x, dim=1, p=0.5):
    """
    shuffles a dimension randomly along dim
    x: [bs x n channels x n patches x patch len]
    """
    if torch.rand(1, device=x.device) > (1-p):
        idx = torch.randperm(x.size(dim), device=x.device)
        return torch.index_select(x, dim, idx)
    else:
        return x


In [ ]:
#| export
def reverse_sequence(x, seq_dim=(-1,), p=0.5):
    if torch.rand(1, device=x.device) > (1-p):
        return torch.flip(x, dims=seq_dim)
    else:
        return x

In [ ]:
#| export
def channel_masking(x, dim=1, p=0.5, specific_channels=None):
    """
    Masks up to n channels - 1 randomly of x or specific channels if provided
    """
    if specific_channels is not None:
        specific_channels = [specific_channels] if isinstance(specific_channels, int) else specific_channels
        idx = torch.tensor(specific_channels, device=x.device)
        if torch.rand(1, device=x.device) > (1-p):
            mask = torch.ones_like(x, device=x.device) # mask of ones
            mask[:,idx] = 0 # set the mask to 0 for the channels to mask
            x = x * mask # apply the mask
    else:
        if torch.rand(1, device=x.device) > (1-p):
            n_channels_to_mask = torch.randint(1, x.size(dim), (1,), device=x.device)
            mask = torch.ones_like(x, device=x.device) # mask of ones
            idx = torch.randperm(x.size(dim), device=x.device)[:n_channels_to_mask] # get n_channels_to_mask random indices
            mask[:,idx] = 0 # set the mask to 0 for the channels to mask
            x = x * mask # apply the mask
    return x

## Transforms

In [ ]:
#| export
class MixupCallbackClassification(object):
    """
    Mixup for 1D data (e.g., time-series).

    This callback applies Mixup to the training data, blending both the input data and the labels.

    See tsai implementation here: https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/data/mixed_augmentation.py#L43

    Note that this creates non-integer labels/soft labels. Loss functions should be able to handle this.
    """
    def __init__(self, 
                 num_classes,
                 mixup_alpha=0.4, # alpha parameter for the beta distribution
                 ignore_index=-100 # ignore index
                 ):
        super().__init__()
        self.distrib = Beta(mixup_alpha, mixup_alpha)
        self.mixup_alpha = mixup_alpha
        self.ignore_index = ignore_index
        self.num_classes = num_classes

    def __call__(self, batch):
        x, y = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        bs, c_in, _ = x.shape
        

        # Sample lambda from a beta distribution
        lam = self.distrib.sample((x.size(0), )).to(x.device) # [bs]
        # our mixing coefficient is always ≥ 0.5
        lam = torch.max(lam, 1 - lam)
        lam_x = lam.view(-1, 1, 1)  # for input shape [bs, channels, seq_len]
        lam_y = lam.view(-1, 1)

        # Shuffle the batch
        indices = torch.randperm(x.size(0), device=x.device)
    
        x_shuffled = x[indices]
        y_shuffled = y[indices]

        # create ignore masks
        ignore_mask = (y == self.ignore_index)
        ignore_mask_shuffled = (y_shuffled == self.ignore_index)
        combined_ignore_mask = torch.logical_or(ignore_mask, ignore_mask_shuffled)

        y_clean = torch.where(ignore_mask, torch.zeros_like(y), y).long()
        y_shuffled_clean = torch.where(ignore_mask_shuffled, torch.zeros_like(y_shuffled), y_shuffled).long()
        # Create one-hot encodings
        if y.ndim == 1:
            y_onehot = F.one_hot(y_clean, num_classes=self.num_classes).float()
            y_shuffled_onehot = F.one_hot(y_shuffled_clean, num_classes=self.num_classes).float()
            ignore_mask = ignore_mask.unsqueeze(-1)
            ignore_mask_shuffled = ignore_mask_shuffled.unsqueeze(-1)
            combined_ignore_mask = combined_ignore_mask.unsqueeze(-1)
        else:
            y_onehot = y_clean.float()
            y_shuffled_onehot = y_shuffled_clean.float()
        # Zero out the one-hot vectors for ignored indices
        y_onehot = torch.where(ignore_mask, torch.zeros_like(y_onehot), y_onehot)
        y_shuffled_onehot = torch.where(ignore_mask_shuffled, torch.zeros_like(y_shuffled_onehot), y_shuffled_onehot)
        # Mixup the inputs and labels
        x_mixed = torch.lerp(x, x_shuffled, lam_x) # x = x_shuffled + lam * (x - x_shuffled)
        y_mixed = torch.lerp(y_onehot, y_shuffled_onehot, lam_y) # y = y_shuffled + lam * (y - y_shuffled)

        # finally assign all probabilities with a -100 mixing to 0
        y_mixed = torch.where(combined_ignore_mask, torch.zeros_like(y_mixed), y_mixed)
        # # add back in the ignore index where it was mixed with other labels
        return (x_mixed, y_mixed)

In [ ]:
#| export
class TransformsCallback(object):
    """
    Applies a series of transforms to the input data, on train_batch_start.
    """
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, batch):
        if len(batch) == 3:
            x, y, time = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        else:
            x, y = batch
        for transform in self.transforms:
            x = transform(x)
        if len(batch) == 3:
            return (x,y,time)
        else:
            return (x,y)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()